# Notebook 1 — Dataset Preparation & Class Formation

**Project:** An Improved Computer Vision Model for Food Classification in Smart Refrigerators using GAN-Based Data Augmentation  
**Author:** Premshakthi Sekar | MSc Artificial Intelligence — Northumbria University  

---

## Purpose
This notebook handles the initial dataset preparation stage. The raw dataset from Roboflow contains refrigerator images with YOLO-format annotation files. This notebook:
1. Reads the annotation CSV file to extract image filenames and class labels
2. Organises images into class-specific subdirectories for use in the CNN classifier

## Dataset Overview
- **Source:** Custom refrigerator food item dataset (collected and annotated via Roboflow)
- **Classes:** 30 food item categories (e.g. apple, chicken, tomato, banana, carrot etc.)
- **Format:** YOLO bounding box annotations (.txt) + images (.jpg)
- **Total images:** ~2,000–3,000 images across train/validation/test splits

## Step 1: Import Libraries

In [ ]:
import os
import shutil
import pandas as pd

print('Libraries imported successfully.')

## Step 2: Load Annotation File

The annotation CSV file maps each image filename to its food class label. This is used to sort images into class folders for the classifier.

In [ ]:
# Path to annotation CSV (update this path to match your local dataset location)
valid_excel_file_path = 'dataset/valid/_annotations.csv'

# Load the annotation file
df = pd.read_csv(valid_excel_file_path)

print(f'Annotation file loaded. Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(5)

## Step 3: Extract Relevant Columns

We only need the filename and class columns. Bounding box coordinates (xmin, ymin, xmax, ymax) and dimensions are not needed for the classification task.

In [ ]:
# Drop unnecessary bounding box and dimension columns
columns_to_drop = ['width', 'height', 'xmin', 'ymin', 'xmax', 'ymax']
df_clean = df.drop(columns_to_drop, axis=1)

print(f'Cleaned dataframe shape: {df_clean.shape}')
print(f'\nUnique food classes ({len(df["class"].unique())} total):')
print(sorted(df['class'].unique()))
df_clean.head(5)

## Step 4: Class Distribution Analysis

Before organising images, we inspect the class distribution to identify any imbalance. Class imbalance is a key motivation for using GAN-based augmentation in later notebooks.

In [ ]:
import matplotlib.pyplot as plt

class_counts = df_clean['class'].value_counts()

plt.figure(figsize=(14, 5))
class_counts.plot(kind='bar', color='#2196F3', edgecolor='navy', alpha=0.8)
plt.title('Class Distribution — Number of Images per Food Category', fontsize=14, fontweight='bold')
plt.xlabel('Food Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMin images per class: {class_counts.min()} ({class_counts.idxmin()})')
print(f'Max images per class: {class_counts.max()} ({class_counts.idxmax()})')
print(f'\nThis class imbalance motivates GAN-based augmentation in Notebook 3.')

## Step 5: Save Class File & Organise Images into Class Folders

Images are moved from a flat directory into class-specific subdirectories. This folder structure is required by the CNN classifier in Notebook 2.

In [ ]:
# Save the cleaned class mapping file
output_class_file = 'dataset/class_mapping.xlsx'
df_clean.to_excel(output_class_file, index=False)
print(f'Class mapping file saved to: {output_class_file}')

In [ ]:
# Paths — update to match your local dataset location
excel_file_path = 'dataset/class_mapping.xlsx'
image_dir = 'dataset/images'

# Load the class mapping
data = pd.read_excel(excel_file_path)

moved = 0
not_found = 0

# Organise each image into its class subfolder
for index, row in data.iterrows():
    image_name = row['filename']
    category = row['class']
    
    source_path = os.path.join(image_dir, image_name)
    
    if os.path.exists(source_path):
        category_dir = os.path.join(image_dir, category)
        os.makedirs(category_dir, exist_ok=True)
        destination_path = os.path.join(category_dir, image_name)
        shutil.move(source_path, destination_path)
        moved += 1
    else:
        not_found += 1

print(f'Images organised successfully.')
print(f'  Moved: {moved}')
print(f'  Not found: {not_found}')
print(f'\nDataset is now ready for the CNN classifier (Notebook 2).')

---
## Summary

| Step | Output |
|------|--------|
| Load annotations | Image filenames mapped to 30 food classes |
| Class distribution | Identified class imbalance motivating GAN augmentation |
| Image organisation | Images sorted into class subdirectories |

**Next:** Notebook 2 — Food Classification Without GAN (Baseline Model)